In [9]:
!pip install vllm

In [10]:
!vllm serve facebook/opt-125m --port 8000 &


2026-03-23 08:50:44.832617: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774255844.855636   11963 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774255844.862487   11963 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774255844.878649   11963 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774255844.878696   11963 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774255844.878701   11963 computation_placer.cc:177] computation placer alr

In [31]:
import subprocess, os

env = os.environ.copy()
env["PATH"] = "/usr/local/bin:" + env["PATH"]

subprocess.Popen(
    ["vllm", "serve", "facebook/opt-125m", "--port", "8000"],
    env=env
)

<Popen: returncode: None args: ['vllm', 'serve', 'facebook/opt-125m', '--por...>

In [32]:
import subprocess
result = subprocess.run(["pgrep", "-f", "vllm"], capture_output=True, text=True)
print(result.stdout)

14605
27846



In [33]:
import time, requests
for i in range(12):
    try:
        r = requests.get("http://localhost:8000/v1/models")
        print(r.json())
        break
    except:
        print(f"محاولة {i+1}... استنى")
        time.sleep(10)

محاولة 1... استنى
محاولة 2... استنى
محاولة 3... استنى
محاولة 4... استنى
محاولة 5... استنى
محاولة 6... استنى
{'object': 'list', 'data': [{'id': 'facebook/opt-125m', 'object': 'model', 'created': 1774259468, 'owned_by': 'vllm', 'root': 'facebook/opt-125m', 'parent': None, 'max_model_len': 2048, 'permission': [{'id': 'modelperm-a5d499618a276d62', 'object': 'model_permission', 'created': 1774259468, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}]}]}


In [ ]:
!pip install aiperf

In [19]:
from google.colab import files
uploaded = files.upload()

Saving BurstGPT_without_fails_3.csv to BurstGPT_without_fails_3.csv


In [20]:
import pandas as pd
df = pd.read_csv("BurstGPT_without_fails_3.csv")
print(df.shape)
print(df.head())

(4956058, 8)
    Timestamp                            Session ID  Elapsed time    Model  \
0  19440110.0  1722ac82-0a46-4bf0-aa08-89794e7a2b3f            43    GPT-4   
1  19440161.0  d5983bf9-4b48-497b-892b-a58995247443             2  ChatGPT   
2  19440192.0  a5c69b35-4fbc-45e9-955e-18715e376d74             8    GPT-4   
3  19440254.0  8b74c7d8-1643-4bba-8c69-91cb4548c506             3  ChatGPT   
4  19440301.0  1722ac82-0a46-4bf0-aa08-89794e7a2b3f            26    GPT-4   

   Request tokens  Response tokens  Total tokens          Log Type  
0             906              446          1352  Conversation log  
1              36               29            65  Conversation log  
2            1779              123          1902  Conversation log  
3             935              178          1113  Conversation log  
4            1631              282          1913  Conversation log  


In [16]:
import subprocess
import json

result = subprocess.run([
    "aiperf", "profile", "facebook/opt-125m",
    "--url", "http://localhost:8000",
    "--endpoint-type", "completions",
    "--streaming",
    "--request-count", "100",
    "--concurrency", "5",
    "--gpu-telemetry", "pynvml",
    "--ui-type", "simple",
    "--output-artifact-dir", "artifacts"
], capture_output=True, text=True)

print(result.stdout)
print(result.stderr)

09:03:39.672 INFO     Starting AIPerf System (cli_runner.py:75)
09:03:40.007 INFO     ✓ Tokenizer facebook/opt-125m detected (tokenizer_display.py:203)
09:03:40.007 INFO     1 tokenizer validated • 0.3s (tokenizer_display.py:212)
09:03:40.263 INFO     Server Metrics: Discovered 1 endpoints: ['http://localhost:8000/metrics'] (manager.py:70)
09:03:40.291 INFO     Registered Server Metrics Manager (id: 'server_metrics_manager') (system_controller.py:340)
09:03:40.309 INFO     GPU telemetry export enabled: artifacts/gpu_telemetry_export.jsonl (jsonl_writer.py:53)
09:03:40.315 INFO     Record metrics export enabled: artifacts/profile_export.jsonl (record_export_results_processor.py:53)
09:03:40.318 INFO     Registered Worker Manager (id: 'worker_manager') (system_controller.py:340)
09:03:40.408 INFO     Registered Gpu Telemetry Manager (id: 'gpu_telemetry_manager') (system_controller.py:340)
09:03:40.421 INFO     Registered Records Manager (id: 'records_manager') (system_controller.py:340)


In [17]:
import requests

PUSHGATEWAY_URL = "https://unrevised-tate-facetiously.ngrok-free.dev"

metrics = """
# HELP ttft_ms Time to first token in milliseconds
# TYPE ttft_ms gauge
ttft_ms 89.25

# HELP itl_ms Inter token latency in milliseconds
# TYPE itl_ms gauge
itl_ms 5.19

# HELP request_latency_ms Request latency in milliseconds
# TYPE request_latency_ms gauge
request_latency_ms 166.69

# HELP throughput_tokens_per_sec Output token throughput
# TYPE throughput_tokens_per_sec gauge
throughput_tokens_per_sec 446.90
"""

headers = {
    "ngrok-skip-browser-warning": "true",
    "Content-Type": "text/plain"
}

r = requests.post(
    f"{PUSHGATEWAY_URL}/metrics/job/aiperf",
    data=metrics,
    headers=headers
)
print(r.status_code)

200


In [28]:
import pandas as pd
import json

df = pd.read_csv("BurstGPT_without_fails_3.csv")
df = df.head(200)

# نرتب حسب الوقت
df = df.sort_values("Timestamp").reset_index(drop=True)

# نحسب الوقت النسبي بالميلي ثانية
first_ts = df["Timestamp"].iloc[0]

with open("burstgpt_trace_timed.jsonl", "w") as f:
    for _, row in df.iterrows():
        entry = {
            "text": "x " * int(row["Request tokens"]),
            "output_length": int(row["Response tokens"]),
            "timestamp": int((row["Timestamp"] - first_ts) * 1000)
        }
        f.write(json.dumps(entry) + "\n")

print(f"تم تحويل {len(df)} طلب مع التوقيت")

تم تحويل 200 طلب مع التوقيت


In [34]:
import subprocess

result = subprocess.run([
    "aiperf", "profile", "facebook/opt-125m",
    "--url", "http://localhost:8000",
    "--endpoint-type", "completions",
    "--streaming",
    "--custom-dataset-type", "single-turn",
    "--input-file", "burstgpt_trace.jsonl",
    "--request-count", "100",
    "--concurrency", "5",
    "--gpu-telemetry", "pynvml",
    "--ui-type", "simple",
    "--output-artifact-dir", "artifacts"
], capture_output=True, text=True)

print(result.stdout)
print(result.stderr)

09:51:25.310 INFO     Starting AIPerf System (cli_runner.py:75)
09:51:25.666 INFO     ✓ Tokenizer facebook/opt-125m detected (tokenizer_display.py:203)
09:51:25.666 INFO     1 tokenizer validated • 0.4s (tokenizer_display.py:212)
09:51:25.954 INFO     Server Metrics: Discovered 1 endpoints: ['http://localhost:8000/metrics'] (manager.py:70)
09:51:25.961 INFO     GPU telemetry export enabled: artifacts/gpu_telemetry_export.jsonl (jsonl_writer.py:53)
09:51:25.971 INFO     Registered Server Metrics Manager (id: 'server_metrics_manager') (system_controller.py:340)
09:51:25.972 INFO     Record metrics export enabled: artifacts/profile_export.jsonl (record_export_results_processor.py:53)
09:51:25.989 INFO     Registered Gpu Telemetry Manager (id: 'gpu_telemetry_manager') (system_controller.py:340)
09:51:25.998 INFO     Registered Dataset Manager (id: 'dataset_manager') (system_controller.py:340)
09:51:25.998 INFO     Registered Worker Manager (id: 'worker_manager') (system_controller.py:340)


In [27]:
import requests

PUSHGATEWAY_URL = "https://unrevised-tate-facetiously.ngrok-free.dev"

metrics = """
# HELP ttft_ms Time to first token in milliseconds
# TYPE ttft_ms gauge
ttft_ms 66.67

# HELP itl_ms Inter token latency in milliseconds
# TYPE itl_ms gauge
itl_ms 5.62

# HELP request_latency_ms Request latency in milliseconds
# TYPE request_latency_ms gauge
request_latency_ms 150.73

# HELP throughput_tokens_per_sec Output token throughput
# TYPE throughput_tokens_per_sec gauge
throughput_tokens_per_sec 468.91
"""

headers = {
    "ngrok-skip-browser-warning": "true",
    "Content-Type": "text/plain"
}

r = requests.post(
    f"{PUSHGATEWAY_URL}/metrics/job/aiperf",
    data=metrics,
    headers=headers
)
print(r.status_code)

200
